In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd

project_path = '/content/drive/MyDrive/Spacecraft-Anomaly-Detection'
data_path = project_path + '/data/raw/archive/data/data'
train_path = data_path + '/train'
test_path = data_path + '/test'
labels_path = project_path + '/data/raw/archive/labeled_anomalies.csv'
results_path = project_path + '/results'

print("Project path:", project_path)
print("Train path exists:", os.path.exists(train_path))
print("Test path exists:", os.path.exists(test_path))
print("Results path exists:", os.path.exists(results_path))

Mounted at /content/drive
Project path: /content/drive/MyDrive/Spacecraft-Anomaly-Detection
Train path exists: True
Test path exists: True
Results path exists: True


In [2]:
channel = "A-8"

train_A8 = np.load(train_path + "/" + channel + ".npy")
test_A8 = np.load(test_path + "/" + channel + ".npy")

train_telemetry = train_A8[:, 0]
test_telemetry = test_A8[:, 0]

labels_df = pd.read_csv(labels_path)

print("Train shape:", train_A8.shape)
print("Test shape:", test_A8.shape)

print("Train telemetry shape:", train_telemetry.shape)
print("Test telemetry shape:", test_telemetry.shape)

print("\nA-8 anomaly information:")
print(labels_df[labels_df["chan_id"] == channel])

Train shape: (762, 25)
Test shape: (8375, 25)
Train telemetry shape: (762,)
Test telemetry shape: (8375,)

A-8 anomaly information:
   chan_id spacecraft anomaly_sequences         class  num_values
52     A-8       SMAP    [[4569, 8374]]  [contextual]        8375


##Set the window size

We'll start with a 50-timestep window.

That means the model will see 50 consecutive telemetry values at a time instead of just one value.

In [3]:
window_size = 50

print("Window size:", window_size)
print("Training points:", len(train_telemetry))
print("Test points:", len(test_telemetry))

Window size: 50
Training points: 762
Test points: 8375


In [4]:
def create_windows(data, window_size):
    windows = []

    for i in range(len(data) - window_size + 1):
        window = data[i:i + window_size]
        windows.append(window)

    return np.array(windows)


train_windows = create_windows(train_telemetry, window_size)
test_windows = create_windows(test_telemetry, window_size)

print("Train windows shape:", train_windows.shape)
print("Test windows shape:", test_windows.shape)

Train windows shape: (713, 50)
Test windows shape: (8326, 50)


This is an important step because now our data has become time sequences rather than individual points

#Create labels for the test windows

Each window ends at a particular timestep. We'll use the last timestep of each window to determine its label.

In [5]:
test_labels = np.zeros(len(test_telemetry), dtype=int)

anomaly_start = 4569
anomaly_end = 8374

test_labels[anomaly_start:anomaly_end + 1] = 1

test_window_labels = test_labels[window_size - 1:]

print("Test window labels shape:", test_window_labels.shape)

print("Normal windows:", np.sum(test_window_labels == 0))
print("Anomaly windows:", np.sum(test_window_labels == 1))

print("Total windows:", len(test_window_labels))

Test window labels shape: (8326,)
Normal windows: 4520
Anomaly windows: 3806
Total windows: 8326


#Check the first and last windows

Before moving to PyTorch/LSTM, let's verify that our windows actually contain the correct telemetry values.

In [6]:
print("First training window:")
print(train_windows[0])

print("\nFirst test window:")
print(test_windows[0])

print("\nLast test window:")
print(test_windows[-1])

print("\nFirst test window label:", test_window_labels[0])
print("Last test window label:", test_window_labels[-1])

First training window:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1.]

First test window:
[0.9392753  0.9392753  0.9392753  0.9392753  0.9392753  0.9392753
 0.9392753  0.9392753  0.9392753  0.9392753  0.9392753  0.9392753
 0.9392753  0.9392753  0.9392753  0.9392753  0.9392753  0.9392753
 0.9392753  0.9392753  1.         0.9392753  0.9392753  0.9392753
 0.9392753  0.9392753  0.9392753  0.9392753  0.9392753  0.9392753
 0.9392753  0.9392753  0.9392753  0.9392753  0.9392753  0.9392753
 0.9392753  0.9392753  0.9392753  0.87855802 0.87855802 0.87855802
 0.87855802 0.87855802 0.87855802 0.81784815 0.81784815 0.81784815
 0.81784815 0.81784815]

Last test window:
[0.33243614 0.33243614 0.33243614 0.33243614 0.33243614 0.33243614
 0.33243614 0.39308669 0.39308669 0.39308669 0.39308669 0.39308669
 0.39308669 0.45374465 0.45374465 0.45374465 0.45374465 0.45374465
 0.45374465 0.45374465 0.5144

The output also shows something interesting about A-8: the telemetry is highly step-like/quantized, with many repeated values. That's consistent with what we observed earlier and is another reason a simple point-wise model struggles.

# Convert windows to PyTorch tensors

Now we're preparing the data for the LSTM.

In [7]:
import torch

X_train = torch.tensor(train_windows, dtype=torch.float32).unsqueeze(-1)
X_test = torch.tensor(test_windows, dtype=torch.float32).unsqueeze(-1)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("X_train dtype:", X_train.dtype)
print("X_test dtype:", X_test.dtype)

X_train shape: torch.Size([713, 50, 1])
X_test shape: torch.Size([8326, 50, 1])
X_train dtype: torch.float32
X_test dtype: torch.float32


#Verify no NaN or infinite values

In [8]:
print("Training NaN:", torch.isnan(X_train).sum().item())
print("Training Inf:", torch.isinf(X_train).sum().item())

print("Test NaN:", torch.isnan(X_test).sum().item())
print("Test Inf:", torch.isinf(X_test).sum().item())

Training NaN: 0
Training Inf: 0
Test NaN: 0
Test Inf: 0


#Save the prepared sequences

Let's save the work so SERIAL 23 can load it without rebuilding everything

In [9]:
sequence_path = results_path + "/time_series_windows_A8.npz"

np.savez(
    sequence_path,
    X_train=X_train.numpy(),
    X_test=X_test.numpy(),
    test_window_labels=test_window_labels,
    window_size=window_size
)

print("Saved:", sequence_path)
print("File exists:", os.path.exists(sequence_path))

Saved: /content/drive/MyDrive/Spacecraft-Anomaly-Detection/results/time_series_windows_A8.npz
File exists: True


#github commit

In [10]:
%cd /content/drive/MyDrive/Spacecraft-Anomaly-Detection

!git status --short

/content/drive/MyDrive/Spacecraft-Anomaly-Detection
Refresh index: 100% (32/32), done.
 M notebooks/13_evaluate_global_zscore.ipynb
 M notebooks/14_rolling_zscore.ipynb
 M notebooks/15_compare_statistical_methods.ipynb
 M notebooks/16_isolation_forest.ipynb
 M notebooks/17_evaluate_isolation_forest.ipynb
 M notebooks/18_one_class_svm.ipynb
 M notebooks/19_compare_classical_methods.ipynb
 M notebooks/20_autoencoder.ipynb
 M notebooks/21_analyze_autoencoder.ipynb
?? notebooks/22_time_series_windows.ipynb
?? results/time_series_windows_A8.npz


In [11]:
!git add notebooks/22_time_series_windows.ipynb results/time_series_windows_A8.npz
!git commit -m "Complete SERIAL 22 time-series windows"

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@0c45084fd7c1.(none)')


In [12]:
!git config --global user.name "Amit Chandra Das"
!git config --global user.email "arickroy0@gmail.com"

print("Git identity configured.")

Git identity configured.


In [13]:
!git add notebooks/22_time_series_windows.ipynb results/time_series_windows_A8.npz
!git commit -m "Complete SERIAL 22 time-series windows"

[main 5c4847c] Complete SERIAL 22 time-series windows
 2 files changed, 1 insertion(+)
 create mode 100644 notebooks/22_time_series_windows.ipynb
 create mode 100644 results/time_series_windows_A8.npz


In [14]:
!git push origin main

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 28.62 KiB | 1.10 MiB/s, done.
Total 6 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Amit-Chandra-Das/spacecraft-anomaly-detection.git
   ae891ce..5c4847c  main -> main
